In [2]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache
from langchain_core.prompts import PromptTemplate

from collections import defaultdict, deque
from datetime import datetime, timedelta
import time

load_dotenv()

# LangChain 캐시 생성
set_llm_cache(InMemoryCache())

# 캐시를 사용하지 않는 모델
llm_no_cache = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    cache=False
)

# 캐시를 사용하는 모델
llm_cache = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    cache=True
)

In [3]:
process_data = """
Lot A: Cycle Time 48 min, Yield 98.5%
Lot B: Cycle Time 52 min, Yield 97.8%
Lot C: Cycle Time 45 min, Yield 99.1%
Lot D: Cycle Time 50 min, Yield 98.3%
"""

prompt = PromptTemplate.from_template(
    """
다음은 공정 데이터입니다.

{data}

질문:
{question}

데이터를 근거로 답변하세요.
"""
)

In [16]:
# 질문별 호출 시간 기록
query_history = defaultdict(deque)

# 실제 캐시
response_cache = {}

# 최근 5분 이내 10회 이상이면 캐시 대상
THRESHOLD = 10
WINDOW_MINUTES = 5

# 캐시 유효시간
CACHE_TTL_MINUTES = 30

In [15]:
def ask_process(question):

    question_key = " ".join(question.strip().split())

    now = datetime.now()

    # ----------------------------
    # 질문 횟수 기록
    # ----------------------------

    history = query_history[question_key]

    limit_time = now - timedelta(minutes=WINDOW_MINUTES)

    while history and history[0] < limit_time:
        history.popleft()

    history.append(now)

    count = len(history)

    use_cache = count >= THRESHOLD


    # ----------------------------
    # Prompt 생성
    # ----------------------------

    final_prompt = prompt.invoke(
        {
            "data": process_data,
            "question": question_key
        }
    )

    start = time.perf_counter()


    # ----------------------------
    # 캐시 존재 여부 확인
    # ----------------------------

    cache_valid = False

    if question_key in response_cache:

        saved_at = response_cache[question_key]["saved_at"]

        cache_age = now - saved_at

        if cache_age < timedelta(minutes=CACHE_TTL_MINUTES):
            cache_valid = True


    # ----------------------------
    # CACHE HIT
    # ----------------------------

    if use_cache and cache_valid:

        response = response_cache[question_key]["response"]

        status = "CACHE HIT"

        actual_tokens = 0


    # ----------------------------
    # 캐시 저장 또는 TTL 만료 후 갱신
    # ----------------------------

    elif use_cache:

        response = llm_no_cache.invoke(final_prompt)

        # 기존 캐시가 있었다면 TTL 만료
        if question_key in response_cache:
            status = "CACHE EXPIRED → REFRESH"

        else:
            status = "CACHE SAVE"

        # 새로운 답변과 시간 저장
        response_cache[question_key] = {
            "response": response,
            "saved_at": now
        }

        actual_tokens = response.usage_metadata.get(
            "total_tokens",
            0
        )


    # ----------------------------
    # 아직 10회 미만
    # ----------------------------

    else:

        response = llm_no_cache.invoke(final_prompt)

        status = "API CALL"

        actual_tokens = response.usage_metadata.get(
            "total_tokens",
            0
        )


    elapsed = time.perf_counter() - start


    # ----------------------------
    # 출력
    # ----------------------------

    print("=" * 60)
    print("질문 :", question_key)
    print("최근 질문 횟수 :", count)
    print("상태 :", status)
    print(f"응답 시간 : {elapsed:.4f}초")
    print("이번 실행 API Token :", actual_tokens)

    # 캐시 정보 출력
    if question_key in response_cache:

        saved_at = response_cache[question_key]["saved_at"]

        age = now - saved_at

        print(
            "캐시 저장 후 경과 시간 :",
            f"{age.total_seconds():.1f}초"
        )

        print(
            "캐시 TTL :",
            f"{CACHE_TTL_MINUTES}분"
        )

    print("=" * 60)

    print(response.content)

In [ ]:
ask_process("평균 Cycle Time과 평균 Yield를 알려줘") 

질문 : 평균 Cycle Time과 평균 Yield를 알려줘
최근 질문 횟수 : 1
상태 : API CALL
응답 시간 : 2.6374초
이번 실행 API Token : 308
캐시 저장 후 경과 시간 : 2374.8초
캐시 TTL : 30분
주어진 데이터를 바탕으로 평균 Cycle Time과 평균 Yield를 계산해 보겠습니다.

1. **Cycle Time 계산:**
   - Lot A: 48분
   - Lot B: 52분
   - Lot C: 45분
   - Lot D: 50분

   평균 Cycle Time = (48 + 52 + 45 + 50) / 4 = 195 / 4 = 48.75분

2. **Yield 계산:**
   - Lot A: 98.5%
   - Lot B: 97.8%
   - Lot C: 99.1%
   - Lot D: 98.3%

   평균 Yield = (98.5 + 97.8 + 99.1 + 98.3) / 4 = 393.7 / 4 = 98.425%

따라서, 평균 Cycle Time은 48.75분이고 평균 Yield는 98.425%입니다.
